# Hybrid QLBM Solver — 2D Convecting Taylor-Green Vortex

**Team Pantheon** · Benjamin Charles Brümm · Vidhi Jain  
Airbus Global Quantum + AI Challenge 2026 · v0.4 milestone

This notebook demonstrates the hybrid quantum-classical QLBM solver for the  
2D Convecting Taylor-Green Vortex (TGV) benchmark.

Sections:
1. Setup — imports and parameters
2. Analytical TGV — exact velocity field at t=0 and t=t_end
3. Hybrid QLBM solver — run on a small (N=8, Re=10) case
4. Velocity error field
5. Benchmark scaling plots

## 1. Setup

In [ ]:
import math
import sys, os
sys.path.insert(0, os.path.join("..", "src"))

import numpy as np
import matplotlib
matplotlib.use("inline")
import matplotlib.pyplot as plt

from tgv.analytical import make_grid, velocity_ic, velocity_exact, l2_error
from tgv.quantum.solver import QLBMSolver, lbm_ic, omega_lbm, n_steps_for_time, dt_physical

# Challenge parameters
Re     = 10.0
N      = 8        # QLBM grid (10 qubits)
t_end  = 0.5      # physical time [s]
V0     = 1.0      # vortex amplitude [m/s]
u_lbm  = 0.05     # LBM velocity scale
L_PHYS = 2*math.pi  # LBM domain size [m]

nu     = V0 * L_PHYS / Re
omega  = omega_lbm(Re, N, u_lbm)
n_steps = n_steps_for_time(t_end, N, u_lbm)
t_actual = n_steps * dt_physical(N, u_lbm, L_PHYS)

print(f"Re={Re}, N={N}, n_qubits=10, omega={omega:.4f}")
print(f"n_steps={n_steps}, t_actual={t_actual:.4f}s")

## 2. Analytical TGV

The exact velocity field uses the challenge parameters (Uc=1, L=2π, V0=1, Re=10).  
Note: `analytical.py` uses L as a length *scale* with domain [0, 2πL].

In [ ]:
L_scale = 2*math.pi  # challenge length scale
x, y = make_grid(64, L_scale)   # 64-point grid for smooth visualization

u0, v0 = velocity_ic(x, y, L=L_scale, V0=V0, Uc=1.0, Vc=0.0)
ut, vt = velocity_exact(x, y, t_actual, L=L_scale, V0=V0, Uc=1.0, Vc=0.0, nu=nu)

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
titles = ['u(x,y,0)', 'v(x,y,0)', f'u(x,y,{t_actual:.2f})', f'v(x,y,{t_actual:.2f})']
fields = [u0, v0, ut, vt]
for ax, field, title in zip(axes.flat, fields, titles):
    im = ax.pcolormesh(x, y, field, cmap='RdBu_r', shading='auto')
    plt.colorbar(im, ax=ax)
    ax.set_title(title)
    ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.set_aspect('equal')
plt.suptitle(f'Analytical TGV — Re={Re}, L={L_scale:.2f}m', fontsize=13)
plt.tight_layout()
plt.show()
print(f"Peak velocity at t=0: {np.max(np.abs(u0)):.4f} m/s")
print(f"Peak velocity at t={t_actual:.2f}: {np.max(np.abs(ut)):.4f} m/s  (decay {np.max(np.abs(ut))/np.max(np.abs(u0))*100:.1f}%)")

## 3. Hybrid QLBM Solver

The QLBM uses quantum streaming (exact unitary circuit) + classical BGK collision.  
In this mode it is bit-for-bit identical to classical LBM.

In [ ]:
solver = QLBMSolver(N=N, omega=omega, u_lbm=u_lbm, use_vqc=False)
f0 = lbm_ic(N, u_lbm)

print(f"Running {n_steps} QLBM steps (N={N}, 10 qubits)...")
f_final = solver.run(f0, n_steps)
print("Done.")

ux_phys, uy_phys = solver.velocity_physical(f_final, V0=V0)
print(f"QLBM peak |u| = {np.max(np.abs(ux_phys)):.4f} m/s")

# Plot QLBM velocity on the coarse (N=8) grid
xs = np.arange(N) / N * 2*math.pi
X8, Y8 = np.meshgrid(xs, xs, indexing='ij')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, field, title in zip(axes, [ux_phys, uy_phys], ['u_QLBM', 'v_QLBM']):
    im = ax.pcolormesh(X8, Y8, field, cmap='RdBu_r', shading='auto')
    plt.colorbar(im, ax=ax)
    ax.set_title(f'{title} at t={t_actual:.2f}s (N={N})')
    ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.set_aspect('equal')
plt.suptitle(f'Hybrid QLBM Output — Re={Re}', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Velocity Error Field

L2 error vs the exact analytical TGV solution.

In [ ]:
# Exact solution on the coarse QLBM grid (lattice-native formula, no Uc)
K = np.arange(N)
K2d, L2d = np.meshgrid(K, K, indexing='ij')
nu_lbm = V0 * L_PHYS / Re
decay = np.exp(-2.0 * nu_lbm * t_actual)
u_exact_grid = V0 * np.sin(2*math.pi*K2d/N) * np.cos(2*math.pi*L2d/N) * decay
v_exact_grid = -V0 * np.cos(2*math.pi*K2d/N) * np.sin(2*math.pi*L2d/N) * decay

err_u = ux_phys - u_exact_grid
err_v = uy_phys - v_exact_grid
err_mag = np.sqrt(err_u**2 + err_v**2)
l2 = float(np.sqrt(np.mean(err_u**2 + err_v**2)))

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, field, title, cmap in zip(axes,
    [err_u, err_v, err_mag],
    ['u_QLBM − u_exact', 'v_QLBM − v_exact', '|error|'],
    ['RdBu_r', 'RdBu_r', 'hot_r']):
    im = ax.pcolormesh(X8, Y8, field, cmap=cmap, shading='auto')
    plt.colorbar(im, ax=ax)
    ax.set_title(title)
    ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.set_aspect('equal')
plt.suptitle(f'QLBM Velocity Error — Re={Re}, t={t_actual:.2f}s, L2={l2:.4f}', fontsize=12)
plt.tight_layout()
plt.show()
print(f"L2 velocity error = {l2:.4f} m/s  ({l2/V0*100:.1f}% of V0)")

## 5. Benchmark Scaling Plots

Generated by `python scripts/sweep.py --full`.

In [ ]:
from IPython.display import Image, display
import os

fig_dir = os.path.join("..", "results", "figures")
plots = [
    ("time_to_solution_vs_re.png", "Time-to-solution vs Re"),
    ("memory_or_qubits_vs_re.png", "Qubit count and memory vs Re"),
    ("l2_error_vs_re.png",         "L2 velocity error vs Re"),
    ("kinetic_energy_decay.png",   "Kinetic energy decay (analytical)"),
]
for fname, caption in plots:
    fpath = os.path.join(fig_dir, fname)
    if os.path.isfile(fpath):
        print(f"\n### {caption}")
        display(Image(fpath))
    else:
        print(f"[{fname} not found — run scripts/sweep.py --full to generate]")